In [ ]:
# workflow enabler
import os, warnings; warnings.filterwarnings("ignore")

# general
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

# PCA
from scipy import stats
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier as HGC
from sklearn.metrics import roc_auc_score
from sklearn.mixture import GaussianMixture as GMM
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

# customized
import hs_med_lib as lib      

# 00 - data input and setup

In [ ]:
# data input
DATA_DIR = "https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data" 
lib.D = DATA_DIR       

dem, trt, sur, asg, tg = lib.load_raw() # tg = target avg_ihs4_next6 (train only)
vis = pd.read_csv(f"{DATA_DIR}/visits.csv", parse_dates=["visit_date"]).sort_values(["patient_id", "visit_date"])
dm = dem.set_index("patient_id")
bio = trt[trt.treatment_class == "Biologic"]; abx = trt[trt.treatment_class == "Antibiotic"]
l1 = bio[bio.treatment_line == 1].set_index("patient_id") # first-line biologic, one row per patient
F = lib.med_features(END, dem, trt) # lifetime treatment table (obs_yrs, abx_rate, ever_bio etc)
F["n_surg"] = sur.groupby("patient_id").size().reindex(F.index).fillna(0); F["surg_rate"] = F.n_surg / F.obs_yrs
F["y"] = tg; F["on_bio_now"] = F.index.isin(bio[bio.end_date.isna()].patient_id).astype(int)
print(f"{len(dm):,} patients | {len(vis):,} visits | {len(trt):,} treatment rows | {len(sur):,} surgeries | data end {END.date()}")

In [ ]:
# ---- Build ONE row per patient, using only information available on END ----
pt = dem.merge(asg, on="patient_id")
pt["dx_year"] = pt.diagnosis_date.dt.year
pt["obs_years"] = (END - pt.diagnosis_date).dt.days / 365.25      # follow-up time since diagnosis

abx = trt[trt.treatment_class == "Antibiotic"]
bio = trt[trt.treatment_class == "Biologic"].sort_values(["patient_id", "start_date"])

def count_by_patient(df, name, mask=None):
    d = df if mask is None else df[mask]
    return d.groupby("patient_id").size().rename(name)

pt = pt.set_index("patient_id")
pt["n_abx"]       = count_by_patient(abx, "n_abx")
pt["n_abx_12m"]   = count_by_patient(abx, "x", abx.start_date > END - pd.Timedelta(days=365))
pt["n_surg"]      = count_by_patient(surg, "n_surg")
pt["n_surg_12m"]  = count_by_patient(surg, "x", surg.procedure_date > END - pd.Timedelta(days=365))
pt["n_bio_lines"] = count_by_patient(bio, "n_bio_lines")
for c in ["n_abx", "n_abx_12m", "n_surg", "n_surg_12m", "n_bio_lines"]:
    pt[c] = pt[c].fillna(0).astype(int)

open_line = bio[bio.end_date.isna()].set_index("patient_id")
pt["on_bio"]       = pt.index.isin(open_line.index)              # open biologic line at END
pt["ever_bio"]     = pt.n_bio_lines > 0
pt["cur_drug"]     = open_line.drug_name
pt["cur_line_no"]  = open_line.treatment_line
pt["days_on_cur"]  = (END - open_line.start_date).dt.days
pt["first_bio_start"] = bio.groupby("patient_id").start_date.min()
pt["bio_status"]   = np.select([pt.on_bio, pt.ever_bio], ["on biologic", "stopped biologic"], "never biologic")

# Rates need a sensible denominator: clip very short follow-up so new patients don't explode
pt["exposure"]  = pt.obs_years.clip(lower=0.25)
pt["abx_rate"]  = pt.n_abx / pt.exposure
pt["surg_rate"] = pt.n_surg / pt.exposure
pt = pt.reset_index()
pt[["patient_id","hurley_stage_baseline","obs_years","n_abx","n_surg","n_bio_lines","bio_status","cur_drug","days_on_cur"]].head()

In [ ]:
# setup
sns.set_theme(style="whitegrid", font_scale=.95); plt.rcParams["figure.dpi"] = 100
HC = {1: "#2a9d8f", 2: "#e9a23b", 3: "#c0392b"}                                  # colour per Hurley stage, same in every plot
TC = {"Deep": "#1b6ca8", "Partial": "#e9a23b", "Minimal": "#9b2d30"}              # colour per response type
MO, END = 30.44, lib.END 

# 01 - data audit

In [ ]:
# data summary
summary = pd.DataFrame({
    "rows":     [len(dem), len(asg), len(sur), len(trt)],
    "patients": [dem.patient_id.nunique(), asg.patient_id.nunique(),
                 sur.patient_id.nunique(), trt.patient_id.nunique()],
}, index=["demographics", "patient_assignment", "surgeries", "treatments"])
summary["share_of_cohort"] = summary.patients / dem.patient_id.nunique()

In [ ]:
# Missingness: how much, and is it related to anything we can see?
fields = ["ethnicity", "bmi_at_baseline", "smoking_status", "symptom_onset_date", "diagnostic_delay_years", "family_history_hs"]
miss = pt[fields].isna()
print("Missing rate per field:\n", miss.mean().round(3).to_string())

# (a) Are the missingness flags correlated with each other? (phi coefficient = Pearson on 0/1)
sns.heatmap(miss.astype(int).corr(), annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-.3, vmax=.3)
plt.title("Correlation between missingness flags"); plt.show()

# (b) Is missingness related to clinical content? Mann-Whitney / chi-square, one test per field x outcome
rows = []
for f in fields:
    flag = miss[f]
    for outcome in ["hurley_stage_baseline", "abx_rate", "surg_rate", "on_bio"]:
        a, b = pt.loc[flag, outcome].astype(float), pt.loc[~flag, outcome].astype(float)
        p = stats.mannwhitneyu(a, b).pvalue
        rows.append((f, outcome, a.mean(), b.mean(), p))
mt = pd.DataFrame(rows, columns=["field", "outcome", "mean_if_missing", "mean_if_present", "p"])
mt["p_bonferroni"] = (mt.p * len(mt)).clip(upper=1)
mt.sort_values("p").head(8)

# 02 - cohort description

In [ ]:
# The cohort summary, split by baseline Hurley stage (the main severity label)
t1 = pt.groupby("hurley_stage_baseline").agg(
    patients=("patient_id", "size"),
    age_at_dx=("age_at_diagnosis", "mean"),
    female=("sex", lambda x: (x == "Female").mean()),
    bmi=("bmi_at_baseline", "mean"),
    current_smoker=("smoking_status", lambda x: (x == "Current").sum() / x.notna().sum()),
    family_hx_yes=("family_history_hs", lambda x: (x == "Yes").sum() / x.isin(["Yes", "No"]).sum()),
    delay_years=("diagnostic_delay_years", "mean"),
    on_biologic=("on_bio", "mean"),
    ever_biologic=("ever_bio", "mean"),
    abx_per_year=("abx_rate", "mean"),
    surgeries_per_year=("surg_rate", "mean"),
    no_surgery=("n_surg", lambda x: (x == 0).mean()),
    untreated=("n_abx", lambda x: ((x == 0)).mean()),
)
t1.T

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.4))

# define yearly data
yr = dem.diagnosis_date.dt.year.value_counts().sort_index(); prev = int((dem.diagnosis_date == lib.T0).sum())

# diagnoses per year
ax = axs[0]; ax.bar(yr.index, yr.values, color="#8fa0b0"); ax.bar(2016, prev, color="#c0392b", label=f"placeholder 2016-01-02 ({prev:,})")
ax.set_title("Diagnoses per calendar year"); ax.set_xlabel("diagnosis year"); ax.legend(fontsize=8, frameon=False)

# share of patients who get bio
ax = axs[1]; g = F.groupby("hurley").agg(patients=("y", "size"), ever_bio=("ever_bio", "mean"), on_bio=("on_bio_now", "mean"))
x = np.arange(3); ax.bar(x - .2, g.ever_bio, .4, color="#5b6b7b", label="ever on a biologic"); ax.bar(x + .2, g.on_bio, .4, color="#1b6ca8", label="on a biologic at end of data")
ax.set_xticks(x); ax.set_xticklabels([f"Hurley {h}\n(n={n:,})" for h, n in g.patients.items()]); ax.set_ylim(0, 1); ax.set_title("Who gets a biologic"); ax.legend(fontsize=8, frameon=False)

# share of patients in each file type
ax = axs[2]; cov = pd.Series({"visits": vis.patient_id.nunique(), "treatments": trt.patient_id.nunique(), "surgeries": sur.patient_id.nunique(), "target": tg.notna().sum(),
                              "images": pd.read_csv(f"{DATA_DIR}/images_metadata.csv").patient_id.nunique(), "RNAseq": pd.read_csv(f"{DATA_DIR}/rnaseq_metadata.csv").patient_id.nunique()}) / len(dm)
ax.barh(cov.index[::-1], cov.values[::-1], color="#8fa0b0"); ax.set_xlim(0, 1); ax.set_title("Share of patients present in each file")

# text position
for i, v in enumerate(cov.values[::-1]): ax.text(v + .01, i, f"{v:.0%}", va="center", fontsize=8)

plt.tight_layout(); plt.show()
print(g.round(3).to_string())

# 02 -- data exploration